**Fact Production**

In [1]:
import pandas as pd
import numpy as np
import random
from datetime import datetime

# ==========================================================
# Manufacturing Analytics
# ViewSonic
# Fact_Production Generator
# ==========================================================

random.seed(42)
np.random.seed(42)

print("="*60)
print("Loading Dimension Tables...")
print("="*60)

# ----------------------------------------------------------
# Load Dimension Tables
# ----------------------------------------------------------

dim_date = pd.read_csv("/content/Dim_Date.csv")
dim_product = pd.read_csv("/content/Dim_Product.csv")
dim_plant = pd.read_csv("/content/Dim_Plant.csv")
dim_machine = pd.read_csv("/content/Dim_Machine.csv")
dim_employee = pd.read_csv("/content/Dim_Employee.csv")
dim_shift = pd.read_csv("/content/Dim_Shift.csv")

print("Date :",len(dim_date))
print("Product :",len(dim_product))
print("Plant :",len(dim_plant))
print("Machine :",len(dim_machine))
print("Employee :",len(dim_employee))
print("Shift :",len(dim_shift))

# ==========================================================
# BUSINESS RULES
# ==========================================================

# ----------------------------------------------------------
# Plant Capacity Weight
# ----------------------------------------------------------

plant_weight = {

1:0.27,
2:0.21,
3:0.16,
4:0.12,
5:0.10,
6:0.07,
7:0.05,
8:0.02

}

# ----------------------------------------------------------
# Product Demand Weight
# ----------------------------------------------------------

product_weight = {

1:0.11,
2:0.10,
3:0.09,
4:0.08,
5:0.07,
6:0.06,
7:0.05,
8:0.05,
9:0.05,
10:0.04,
11:0.04,
12:0.04,
13:0.03,
14:0.03,
15:0.03,
16:0.03,
17:0.02,
18:0.02,
19:0.01,
20:0.01,
21:0.01,
22:0.01

}

# ----------------------------------------------------------
# Monthly Seasonality
# ----------------------------------------------------------

season_factor = {

1:0.82,
2:0.86,
3:0.92,
4:0.97,
5:1.04,
6:1.11,
7:1.18,
8:1.15,
9:1.01,
10:1.08,
11:1.24,
12:1.36

}

# ----------------------------------------------------------
# Shift Productivity
# ----------------------------------------------------------

shift_factor = {

1:1.00,
2:0.96,
3:0.90

}

# ----------------------------------------------------------
# Shift Defect Rate
# ----------------------------------------------------------

shift_defect = {

1:0.020,
2:0.025,
3:0.035

}

# ----------------------------------------------------------
# Plant Specialization
# ----------------------------------------------------------

plant_products = {

1:[1,2,3,4,5],
2:[6,7,8],
3:[9,10,11,12],
4:[13,14,15],
5:[16,17,18],
6:[19,20],
7:[21],
8:[22]

}

# ==========================================================
# Create Employee Lookup
# ==========================================================

employee_salary = dict(
zip(
dim_employee.EmployeeKey,
dim_employee.SalaryUSD
)
)

employee_exp = dict(
zip(
dim_employee.EmployeeKey,
dim_employee.ExperienceYears
)
)

# ==========================================================
# Product Lookup
# ==========================================================

unit_cost = dict(
zip(
dim_product.ProductKey,
dim_product.UnitCost
)
)

unit_price = dict(
zip(
dim_product.ProductKey,
dim_product.UnitPrice
)
)

print("\nBusiness Rules Loaded Successfully")
print("="*60)

Loading Dimension Tables...
Date : 1096
Product : 22
Plant : 8
Machine : 38
Employee : 200
Shift : 3

Business Rules Loaded Successfully


In [2]:
# ==========================================================
# Generate Production Transactions
# ==========================================================

production = []

production_id = 1

print("\nGenerating Production Data...")

for _, date_row in dim_date.iterrows():

    date_key = int(date_row["DateKey"])
    month = pd.to_datetime(date_row["Date"]).month
    season = season_factor[month]

    # Loop through every plant
    for plant in plant_weight.keys():

        # Number of production batches depends on plant size
        if plant == 1:
            batches = random.randint(7,10)
        elif plant == 2:
            batches = random.randint(6,9)
        elif plant == 3:
            batches = random.randint(5,8)
        elif plant == 4:
            batches = random.randint(4,7)
        elif plant == 5:
            batches = random.randint(4,6)
        elif plant == 6:
            batches = random.randint(3,5)
        elif plant == 7:
            batches = random.randint(2,4)
        else:
            batches = random.randint(1,3)

        # Machines available in plant
        machine_list = dim_machine[
            dim_machine["PlantKey"] == plant
        ]["MachineKey"].tolist()

        # Employees available in plant
        employee_list = dim_employee[
            dim_employee["PlantKey"] == plant
        ]["EmployeeKey"].tolist()

        # Products manufactured in plant
        product_list = plant_products[plant]

        # Product demand weights
        weights = [
            product_weight[p]
            for p in product_list
        ]

        total = sum(weights)
        weights = [w/total for w in weights]

        for i in range(batches):

            shift = random.choice([1,2,3])

            machine = random.choice(machine_list)

            employee = random.choice(employee_list)

            product = np.random.choice(
                product_list,
                p=weights
            )

            # Production hours
            production_hours = round(
                random.uniform(6.5,8.0),
                2
            )

            # Machine utilization
            utilization = round(
                random.uniform(82,98)
                * shift_factor[shift],
                2
            )

            # Downtime
            downtime = random.randint(5,60)

            # Machine capacity
            capacity = int(
                dim_machine.loc[
                    dim_machine["MachineKey"]==machine,
                    "CapacityPerHour"
                ].values[0]
            )

            # Employee experience
            exp = employee_exp[employee]

            exp_factor = min(
                1.12,
                0.90 + exp*0.01
            )

            # Quantity Produced
            quantity = int(
                capacity
                * production_hours
                * season
                * plant_weight[plant]
                * exp_factor
                * random.uniform(0.90,1.08)
            )

            quantity = max(quantity,50)

            production.append({

                "ProductionID":production_id,

                "DateKey":date_key,

                "PlantKey":plant,

                "MachineKey":machine,

                "EmployeeKey":employee,

                "ProductKey":product,

                "ShiftKey":shift,

                "ProductionHours":production_hours,

                "MachineUtilizationPct":utilization,

                "DowntimeMinutes":downtime,

                "QuantityProduced":quantity

            })

            production_id += 1

print("Production Transactions :",len(production))

production_df = pd.DataFrame(production)


Generating Production Data...
Production Transactions : 46037


In [3]:
# ==========================================================
# Production KPI Calculation
# ==========================================================

print("\nCalculating KPIs...")

good_qty = []
reject_qty = []
defect_pct = []
oee_list = []

gross_revenue = []

production_cost = []
downtime_cost = []
labor_cost = []
scrap_cost = []

total_cost = []

profit = []

profit_margin = []

cost_per_good = []

for _, row in production_df.iterrows():

    qty = row["QuantityProduced"]

    shift = row["ShiftKey"]

    emp = row["EmployeeKey"]

    machine = row["MachineKey"]

    product = row["ProductKey"]

    hours = row["ProductionHours"]

    downtime = row["DowntimeMinutes"]

    util = row["MachineUtilizationPct"]

    # ---------------------------------------------------
    # Employee Experience
    # ---------------------------------------------------

    exp = employee_exp[emp]

    experience_bonus = min(0.008, exp * 0.0003)

    # ---------------------------------------------------
    # Defect Rate
    # ---------------------------------------------------

    defect = shift_defect[shift]

    defect = defect + random.uniform(-0.004,0.004)

    defect = defect - experience_bonus

    defect = max(0.01, defect)

    defect = min(0.06, defect)

    reject = round(qty * defect)

    good = qty - reject

    # ---------------------------------------------------
    # OEE
    # ---------------------------------------------------

    availability = max(
        0.70,
        1 - downtime/(hours*60)
    )

    performance = util/100

    quality = good/qty

    oee = availability * performance * quality * 100

    # ---------------------------------------------------
    # Revenue
    # ---------------------------------------------------

    price = unit_price[product]

    revenue = good * price

    # ---------------------------------------------------
    # Costs
    # ---------------------------------------------------

    cost = qty * unit_cost[product]

    dt_cost = downtime * random.uniform(8,15)

    salary = employee_salary[emp]

    labour = (salary/2080) * hours

    scrap = reject * unit_cost[product]

    total = (
        cost
        + dt_cost
        + labour
        + scrap
    )

    pr = revenue - total

    margin = 0

    if revenue > 0:
        margin = (pr/revenue)*100

    cpu = total/max(good,1)

    # ---------------------------------------------------

    good_qty.append(good)

    reject_qty.append(reject)

    defect_pct.append(round(defect*100,2))

    oee_list.append(round(oee,2))

    gross_revenue.append(round(revenue,2))

    production_cost.append(round(cost,2))

    downtime_cost.append(round(dt_cost,2))

    labor_cost.append(round(labour,2))

    scrap_cost.append(round(scrap,2))

    total_cost.append(round(total,2))

    profit.append(round(pr,2))

    profit_margin.append(round(margin,2))

    cost_per_good.append(round(cpu,2))

# ==========================================================
# Append Calculated Columns
# ==========================================================

production_df["GoodQuantity"] = good_qty
production_df["RejectedQuantity"] = reject_qty
production_df["DefectRatePct"] = defect_pct
production_df["OEE"] = oee_list

production_df["UnitCost"] = production_df["ProductKey"].map(unit_cost)
production_df["UnitPrice"] = production_df["ProductKey"].map(unit_price)

production_df["GrossRevenue"] = gross_revenue
production_df["ProductionCost"] = production_cost
production_df["DowntimeCost"] = downtime_cost
production_df["LaborCost"] = labor_cost
production_df["ScrapCost"] = scrap_cost

production_df["TotalCost"] = total_cost

production_df["Profit"] = profit
production_df["ProfitMarginPct"] = profit_margin
production_df["CostPerGoodUnit"] = cost_per_good

print("KPI Calculation Completed")


Calculating KPIs...
KPI Calculation Completed


In [4]:
# ==========================================================
# Arrange Columns (Must Match SQL Table)
# ==========================================================

production_df = production_df[[
    "ProductionID",
    "DateKey",
    "PlantKey",
    "MachineKey",
    "EmployeeKey",
    "ProductKey",
    "ShiftKey",
    "ProductionHours",
    "MachineUtilizationPct",
    "DowntimeMinutes",
    "QuantityProduced",
    "GoodQuantity",
    "RejectedQuantity",
    "DefectRatePct",
    "OEE",
    "UnitCost",
    "UnitPrice",
    "GrossRevenue",
    "ProductionCost",
    "DowntimeCost",
    "LaborCost",
    "ScrapCost",
    "TotalCost",
    "Profit",
    "ProfitMarginPct",
    "CostPerGoodUnit"
]]

# ==========================================================
# Data Validation
# ==========================================================

assert production_df.isnull().sum().sum() == 0, "Null values found."

assert (production_df["QuantityProduced"] >= production_df["GoodQuantity"]).all()

assert (
    production_df["QuantityProduced"]
    ==
    production_df["GoodQuantity"]
    + production_df["RejectedQuantity"]
).all()

assert (production_df["GrossRevenue"] >= 0).all()
assert (production_df["TotalCost"] >= 0).all()

# ==========================================================
# Summary
# ==========================================================

print("\n" + "="*70)
print("FACT PRODUCTION SUMMARY")
print("="*70)

print("Rows :", len(production_df))
print("Columns :", len(production_df.columns))

print("\nTotal Production :",
      format(int(production_df["QuantityProduced"].sum()),","))

print("Good Quantity :",
      format(int(production_df["GoodQuantity"].sum()),","))

print("Rejected Quantity :",
      format(int(production_df["RejectedQuantity"].sum()),","))

print("\nRevenue : $",
      format(round(production_df["GrossRevenue"].sum(),2),","))

print("Profit : $",
      format(round(production_df["Profit"].sum(),2),","))

print("\nAverage OEE :",
      round(production_df["OEE"].mean(),2),"%")

print("Average Defect Rate :",
      round(production_df["DefectRatePct"].mean(),2),"%")

print("="*70)

# ==========================================================
# Export CSV
# ==========================================================

production_df.to_csv(
    "Fact_Production.csv",
    index=False
)

print("\nFact_Production.csv Created Successfully")

print("\nPreview\n")

print(production_df.head(10))


FACT PRODUCTION SUMMARY
Rows : 46037
Columns : 26

Total Production : 8,986,332
Good Quantity : 8,783,419
Rejected Quantity : 202,913

Revenue : $ 5,363,275,525.0
Profit : $ 2,276,413,395.02

Average OEE : 77.61 %
Average Defect Rate : 2.27 %

Fact_Production.csv Created Successfully

Preview

   ProductionID   DateKey  PlantKey  MachineKey  EmployeeKey  ProductKey  \
0             1  20230101         1           6           86           2   
1             2  20230101         1           1          156           5   
2             3  20230101         1           1          145           4   
3             4  20230101         1           5           86           3   
4             5  20230101         1           3           86           1   
5             6  20230101         1           1          102           1   
6             7  20230101         1           5           30           1   
7             8  20230101         2           8           13           8   
8             9  202

**Fact Inventory**

In [5]:
import pandas as pd
import numpy as np
import random

# ==========================================================
# Manufacturing Analytics
# ViewSonic
# Fact_Inventory Generator
# ==========================================================

random.seed(42)
np.random.seed(42)

print("=" * 60)
print("Loading Files...")
print("=" * 60)

# -----------------------------
# Load CSV Files
# -----------------------------

production = pd.read_csv("/content/Fact_Production.csv")

dim_product = pd.read_csv("/content/Dim_Product.csv")

dim_supplier = pd.read_csv("/content/Dim_Supplier.csv")

dim_warehouse = pd.read_csv("/content/Dim_Warehouse.csv")

dim_date = pd.read_csv("/content/Dim_Date.csv")

print("Production :", len(production))
print("Products :", len(dim_product))
print("Suppliers :", len(dim_supplier))
print("Warehouses :", len(dim_warehouse))

# ==========================================================
# Business Rules
# ==========================================================

warehouse_capacity = dict(
    zip(
        dim_warehouse["WarehouseKey"],
        dim_warehouse["CapacityUnits"]
    )
)

# Product -> Warehouse Mapping

warehouse_map = {
    1:1,2:1,3:1,4:1,
    5:2,6:2,7:2,8:2,
    9:3,10:3,11:3,12:3,
    13:4,14:4,15:4,16:4,
    17:5,18:5,19:5,
    20:6,21:6,22:6
}

# Product -> Supplier Mapping

supplier_map = {
    1:1,2:1,3:2,4:2,
    5:3,6:3,7:4,8:4,
    9:5,10:5,11:6,12:6,
    13:7,14:8,15:9,16:10,
    17:11,18:12,19:13,
    20:14,21:15,22:16
}

# Product Unit Cost

unit_cost = dict(
    zip(
        dim_product["ProductKey"],
        dim_product["UnitCost"]
    )
)

inventory = []

inventory_id = 1

print("\nBusiness Rules Loaded")

Loading Files...
Production : 46037
Products : 22
Suppliers : 17
Warehouses : 6

Business Rules Loaded


In [6]:
# ==========================================================
# Generate Inventory Records
# ==========================================================

print("\nGenerating Inventory Data...")

# Group production by Date & Product
prod_summary = (
    production
    .groupby(["DateKey", "ProductKey"])["QuantityProduced"]
    .sum()
    .reset_index()
)

stock_balance = {}

for _, row in prod_summary.iterrows():

    date_key = int(row["DateKey"])
    product = int(row["ProductKey"])

    warehouse = warehouse_map[product]
    supplier = supplier_map[product]

    produced = int(row["QuantityProduced"])

    # --------------------------------------------
    # First Day Opening Stock
    # --------------------------------------------

    if product not in stock_balance:

        opening = random.randint(
            produced * 2,
            produced * 5
        )

    else:

        opening = stock_balance[product]

    # --------------------------------------------
    # Stock Received
    # --------------------------------------------

    received = int(
        produced *
        random.uniform(0.92,1.08)
    )

    # --------------------------------------------
    # Daily Demand
    # --------------------------------------------

    demand = int(
        produced *
        random.uniform(0.75,0.95)
    )

    # --------------------------------------------
    # Stock Issued
    # --------------------------------------------

    issued = demand

    # --------------------------------------------
    # Closing Stock
    # --------------------------------------------

    closing = opening + received - issued

    # Never Negative

    closing = max(closing,0)

    stock_balance[product] = closing

    # --------------------------------------------
    # Safety Stock
    # --------------------------------------------

    safety = int(
        demand * 7
    )

    # --------------------------------------------
    # Reorder Level
    # --------------------------------------------

    reorder = int(
        safety * 1.35
    )

    # --------------------------------------------
    # Inventory Status
    # --------------------------------------------

    if closing < safety:

        status = "Critical"

    elif closing < reorder:

        status = "Reorder"

    else:

        status = "Healthy"

    # --------------------------------------------
    # Inventory Value
    # --------------------------------------------

    inv_value = round(
        closing *
        unit_cost[product],
        2
    )

    # --------------------------------------------
    # Inventory Turnover
    # --------------------------------------------

    avg_inventory = (opening + closing) / 2

    turnover = round(
        issued /
        max(avg_inventory,1),
        2
    )

    # --------------------------------------------
    # Days Of Inventory
    # --------------------------------------------

    doi = round(
        closing /
        max(demand,1),
        2
    )

    # --------------------------------------------
    # Warehouse Utilization
    # --------------------------------------------

    utilization = round(
        (closing /
         warehouse_capacity[warehouse])
         *100,
         2
    )

    utilization = min(utilization,99.5)

    # --------------------------------------------
    # Carrying Cost
    # --------------------------------------------

    carrying = round(
        inv_value * 0.015,
        2
    )

    # --------------------------------------------
    # Stock Age
    # --------------------------------------------

    age = random.randint(5,90)

    # --------------------------------------------
    # Expired Inventory
    # --------------------------------------------

    expired = 0

    if age > 75:

        expired = random.randint(
            0,
            max(2,int(closing*0.01))
        )

    inventory.append({

        "InventoryID":inventory_id,

        "DateKey":date_key,

        "ProductKey":product,

        "WarehouseKey":warehouse,

        "SupplierKey":supplier,

        "OpeningStock":opening,

        "StockReceived":received,

        "StockIssued":issued,

        "ClosingStock":closing,

        "ReorderLevel":reorder,

        "SafetyStock":safety,

        "InventoryValue":inv_value,

        "InventoryStatus":status,

        "DailyDemand":demand,

        "InventoryTurnover":turnover,

        "DaysOfInventory":doi,

        "WarehouseUtilizationPct":utilization,

        "CarryingCost":carrying,

        "StockAgeDays":age,

        "ExpiredInventory":expired

    })

    inventory_id += 1

print("Inventory Records :",len(inventory))

inventory_df = pd.DataFrame(inventory)


Generating Inventory Data...
Inventory Records : 21248


In [7]:
# ==========================================================
# Arrange Columns (Match SQL Table)
# ==========================================================

inventory_df = inventory_df[[
    "InventoryID",
    "DateKey",
    "ProductKey",
    "WarehouseKey",
    "SupplierKey",
    "OpeningStock",
    "StockReceived",
    "StockIssued",
    "ClosingStock",
    "ReorderLevel",
    "SafetyStock",
    "InventoryValue",
    "InventoryStatus",
    "DailyDemand",
    "InventoryTurnover",
    "DaysOfInventory",
    "WarehouseUtilizationPct",
    "CarryingCost",
    "StockAgeDays",
    "ExpiredInventory"
]]

# ==========================================================
# Validation
# ==========================================================

assert inventory_df.isnull().sum().sum() == 0

assert (
    inventory_df["OpeningStock"]
    + inventory_df["StockReceived"]
    - inventory_df["StockIssued"]
    ==
    inventory_df["ClosingStock"]
).all()

assert (inventory_df["ClosingStock"] >= 0).all()

assert (inventory_df["InventoryValue"] >= 0).all()

assert (inventory_df["WarehouseUtilizationPct"] <= 100).all()

# ==========================================================
# Summary
# ==========================================================

print("\n" + "="*70)
print("FACT INVENTORY SUMMARY")
print("="*70)

print("Rows :", len(inventory_df))
print("Columns :", len(inventory_df.columns))

print("\nTotal Opening Stock :",
      format(int(inventory_df["OpeningStock"].sum()),","))

print("Total Closing Stock :",
      format(int(inventory_df["ClosingStock"].sum()),","))

print("Inventory Value : $",
      format(round(inventory_df["InventoryValue"].sum(),2),","))

print("Average Warehouse Utilization :",
      round(inventory_df["WarehouseUtilizationPct"].mean(),2),
      "%")

print("Average Inventory Turnover :",
      round(inventory_df["InventoryTurnover"].mean(),2))

print("="*70)

# ==========================================================
# Export CSV
# ==========================================================

inventory_df.to_csv(
    "Fact_Inventory.csv",
    index=False
)

print("\nFact_Inventory.csv Created Successfully")

print("\nPreview\n")

print(inventory_df.head(10))


FACT INVENTORY SUMMARY
Rows : 21248
Columns : 20

Total Opening Stock : 649,122,495
Total Closing Stock : 650,461,326
Inventory Value : $ 217,868,282,307
Average Warehouse Utilization : 26.19 %
Average Inventory Turnover : 0.03

Fact_Inventory.csv Created Successfully

Preview

   InventoryID   DateKey  ProductKey  WarehouseKey  SupplierKey  OpeningStock  \
0            1  20230101           1             1            1          1986   
1            2  20230101           2             1            1          1003   
2            3  20230101           3             1            2           521   
3            4  20230101           4             1            2           713   
4            5  20230101           5             2            3          1229   
5            6  20230101           6             2            3          2143   
6            7  20230101           7             2            4           541   
7            8  20230101           8             2            4         

**Fact Procurement**

In [8]:
import pandas as pd
import numpy as np
import random

# ==========================================================
# Manufacturing Analytics
# ViewSonic
# Fact_Procurement Generator
# ==========================================================

random.seed(42)
np.random.seed(42)

print("="*60)
print("Loading Files...")
print("="*60)

inventory = pd.read_csv("/content/Fact_Inventory.csv")

dim_supplier = pd.read_csv("/content/Dim_Supplier.csv")

dim_product = pd.read_csv("/content/Dim_Product.csv")

dim_plant = pd.read_csv("/content/Dim_Plant.csv")

print("Inventory :",len(inventory))
print("Suppliers :",len(dim_supplier))
print("Products :",len(dim_product))

# ==========================================================
# Lookup Tables
# ==========================================================

supplier_rating = dict(
    zip(
        dim_supplier.SupplierKey,
        dim_supplier.Rating
    )
)

supplier_leadtime = dict(
    zip(
        dim_supplier.SupplierKey,
        dim_supplier.LeadTimeDays
    )
)

unit_cost = dict(
    zip(
        dim_product.ProductKey,
        dim_product.UnitCost
    )
)

# Product -> Plant Mapping

plant_map = {

1:1,2:1,3:1,4:1,
5:2,6:2,7:2,8:2,
9:3,10:3,11:3,12:3,
13:4,14:4,15:4,
16:5,17:5,18:5,
19:6,20:6,
21:7,
22:8

}

procurement=[]

procurement_id=1

print("\nBusiness Rules Loaded")

Loading Files...
Inventory : 21248
Suppliers : 17
Products : 22

Business Rules Loaded


In [9]:
print("\nGenerating Procurement Data...")

for _, row in inventory.iterrows():

    opening = row["OpeningStock"]
    closing = row["ClosingStock"]
    reorder = row["ReorderLevel"]

    product = int(row["ProductKey"])

    supplier = int(row["SupplierKey"])

    plant = plant_map[product]

    date = int(row["DateKey"])

    # ----------------------------------------

    if closing < reorder:

        ordered = random.randint(

            reorder*2,

            reorder*4

        )

    else:

        ordered = random.randint(

            int(reorder*0.20),

            int(reorder*0.60)

        )

    # ----------------------------------------

    fill_rate = random.uniform(95,100)

    received = int(

        ordered *

        fill_rate/100

    )

    variance = ordered-received

    price = unit_cost[product]

    purchase = received*price

    freight = purchase*random.uniform(0.02,0.05)

    tax = purchase*0.08

    total = purchase+freight+tax

    lead = supplier_leadtime[supplier]

    delay=max(0,int(np.random.normal(1,2)))

    acceptance=random.uniform(96,100)

    performance=(

        acceptance*0.40+

        fill_rate*0.35+

        (100-delay*5)*0.25

    )

    if received==ordered:

        status="Completed"

    elif received>0:

        status="Partial"

    else:

        status="Pending"

    payment=random.choice([

        "Paid",

        "Pending"

    ])

    procurement.append({

        "ProcurementID":procurement_id,

        "PONumber":f"PO-{2023+(procurement_id//10000)}-{procurement_id:06}",

        "DateKey":date,

        "SupplierKey":supplier,

        "ProductKey":product,

        "PlantKey":plant,

        "OrderedQuantity":ordered,

        "ReceivedQuantity":received,

        "QuantityVariance":variance,

        "UnitCost":price,

        "PurchaseAmount":round(purchase,2),

        "FreightCost":round(freight,2),

        "TaxAmount":round(tax,2),

        "TotalProcurementCost":round(total,2),

        "LeadTimeDays":lead,

        "DeliveryDelayDays":delay,

        "FillRatePct":round(fill_rate,2),

        "QualityAcceptancePct":round(acceptance,2),

        "SupplierRating":supplier_rating[supplier],

        "SupplierPerformanceScore":round(performance,2),

        "OrderStatus":status,

        "PaymentStatus":payment

    })

    procurement_id+=1

procurement_df=pd.DataFrame(procurement)

print("Procurement Orders :",len(procurement_df))


Generating Procurement Data...
Procurement Orders : 21248


In [10]:
# ==========================================================
# Arrange Columns (Match SQL Table)
# ==========================================================

procurement_df = procurement_df[[
    "ProcurementID",
    "PONumber",
    "DateKey",
    "SupplierKey",
    "ProductKey",
    "PlantKey",
    "OrderedQuantity",
    "ReceivedQuantity",
    "QuantityVariance",
    "UnitCost",
    "PurchaseAmount",
    "FreightCost",
    "TaxAmount",
    "TotalProcurementCost",
    "LeadTimeDays",
    "DeliveryDelayDays",
    "FillRatePct",
    "QualityAcceptancePct",
    "SupplierRating",
    "SupplierPerformanceScore",
    "OrderStatus",
    "PaymentStatus"
]]

# ==========================================================
# Validation
# ==========================================================

assert procurement_df.isnull().sum().sum() == 0

assert (
    procurement_df["OrderedQuantity"]
    >=
    procurement_df["ReceivedQuantity"]
).all()

assert (
    procurement_df["QuantityVariance"]
    ==
    procurement_df["OrderedQuantity"]
    -
    procurement_df["ReceivedQuantity"]
).all()

assert (
    procurement_df["TotalProcurementCost"]
    >=
    procurement_df["PurchaseAmount"]
).all()

# ==========================================================
# Summary
# ==========================================================

print("\n" + "="*70)
print("FACT PROCUREMENT SUMMARY")
print("="*70)

print("Rows :", len(procurement_df))
print("Columns :", len(procurement_df.columns))

print("\nTotal Purchase Amount : $",
      format(round(procurement_df["PurchaseAmount"].sum(),2),","))

print("Total Procurement Cost : $",
      format(round(procurement_df["TotalProcurementCost"].sum(),2),","))

print("Average Fill Rate :",
      round(procurement_df["FillRatePct"].mean(),2),"%")

print("Average Supplier Rating :",
      round(procurement_df["SupplierRating"].mean(),2))

print("Average Supplier Performance :",
      round(procurement_df["SupplierPerformanceScore"].mean(),2))

print("="*70)

# ==========================================================
# Export CSV
# ==========================================================

procurement_df.to_csv(
    "Fact_Procurement.csv",
    index=False
)

print("\nFact_Procurement.csv Created Successfully")

print("\nPreview\n")

print(procurement_df.head(10))


FACT PROCUREMENT SUMMARY
Rows : 21248
Columns : 22

Total Purchase Amount : $ 12,193,716,034
Total Procurement Cost : $ 13,595,434,340.76
Average Fill Rate : 97.5 %
Average Supplier Rating : 4.74
Average Supplier Performance : 96.98

Fact_Procurement.csv Created Successfully

Preview

   ProcurementID        PONumber   DateKey  SupplierKey  ProductKey  PlantKey  \
0              1  PO-2023-000001  20230101            1           1         1   
1              2  PO-2023-000002  20230101            1           2         1   
2              3  PO-2023-000003  20230101            2           3         1   
3              4  PO-2023-000004  20230101            2           4         1   
4              5  PO-2023-000005  20230101            3           5         2   
5              6  PO-2023-000006  20230101            3           6         2   
6              7  PO-2023-000007  20230101            4           7         2   
7              8  PO-2023-000008  20230101            4          

**Fact Quality**

In [11]:
import pandas as pd
import numpy as np
import random

# ==========================================================
# Manufacturing Analytics
# ViewSonic
# Fact_Quality Generator
# ==========================================================

random.seed(42)
np.random.seed(42)

print("="*60)
print("Loading Files...")
print("="*60)

production = pd.read_csv("/content/Fact_Production.csv")

employee = pd.read_csv("/content/Dim_Employee.csv")

print("Production :",len(production))

quality=[]

quality_id=1

# Employee Experience

employee_exp=dict(
zip(
employee.EmployeeKey,
employee.ExperienceYears
)
)

# Defect Types

defect_types=[
"Scratch",
"Dead Pixel",
"Loose Connection",
"Color Issue",
"Alignment Error",
"Display Flicker",
"Packaging Damage",
"Power Failure"
]

print("\nGenerating Quality Data...")

Loading Files...
Production : 46037

Generating Quality Data...


In [12]:
for _,row in production.iterrows():

    inspected=row["QuantityProduced"]

    passed=row["GoodQuantity"]

    defective=row["RejectedQuantity"]

    defect=row["DefectRatePct"]

    exp=employee_exp[row["EmployeeKey"]]

    inspection_hours=round(
        inspected/500+
        random.uniform(0.5,2),
        2
    )

    inspector_score=round(

        min(
            100,

            82+

            exp*0.7+

            random.uniform(-2,5)

        ),

        2

    )

    rework=int(

        defective*

        random.uniform(0.35,0.70)

    )

    scrap=defective-rework

    fpypct=round(

        (passed/inspected)*100,

        2

    )

    reworkpct=round(

        (rework/max(defective,1))*100,

        2

    )

    scrappct=round(

        (scrap/max(defective,1))*100,

        2

    )

    rework_cost=round(

        rework*

        random.uniform(6,18),

        2

    )

    scrap_cost=round(

        scrap*

        row["UnitCost"],

        2

    )

    total_cost=round(

        rework_cost+

        scrap_cost,

        2

    )

    if defect<2:

        result="Pass"

    elif defect<4:

        result="Conditional Pass"

    else:

        result="Fail"

    quality.append({

        "QualityID":quality_id,

        "DateKey":row["DateKey"],

        "ProductKey":row["ProductKey"],

        "MachineKey":row["MachineKey"],

        "EmployeeKey":row["EmployeeKey"],

        "ShiftKey":row["ShiftKey"],

        "PlantKey":row["PlantKey"],

        "InspectedQuantity":inspected,

        "PassedQuantity":passed,

        "DefectiveQuantity":defective,

        "DefectRatePct":defect,

        "InspectionResult":result,

        "DefectType":random.choice(defect_types),

        "InspectionTimeHours":inspection_hours,

        "InspectorScore":inspector_score,

        "ReworkQuantity":rework,

        "ScrapQuantity":scrap,

        "FirstPassYieldPct":fpypct,

        "ReworkRatePct":reworkpct,

        "ScrapRatePct":scrappct,

        "ReworkCost":rework_cost,

        "ScrapCost":scrap_cost,

        "TotalQualityCost":total_cost

    })

    quality_id+=1

quality_df=pd.DataFrame(quality)

print("Quality Records :",len(quality_df))

Quality Records : 46037


In [13]:
quality_df=quality_df[[

"QualityID",
"DateKey",
"ProductKey",
"MachineKey",
"EmployeeKey",
"ShiftKey",
"PlantKey",
"InspectedQuantity",
"PassedQuantity",
"DefectiveQuantity",
"DefectRatePct",
"InspectionResult",
"DefectType",
"InspectionTimeHours",
"InspectorScore",
"ReworkQuantity",
"ScrapQuantity",
"FirstPassYieldPct",
"ReworkRatePct",
"ScrapRatePct",
"ReworkCost",
"ScrapCost",
"TotalQualityCost"

]]

assert quality_df.isnull().sum().sum()==0

assert (
quality_df["PassedQuantity"]+
quality_df["DefectiveQuantity"]
==
quality_df["InspectedQuantity"]
).all()

quality_df.to_csv(
"Fact_Quality.csv",
index=False
)

print("="*60)
print("Fact_Quality.csv Created Successfully")
print("="*60)

print(quality_df.head())

Fact_Quality.csv Created Successfully
   QualityID     DateKey  ProductKey  MachineKey  EmployeeKey  ShiftKey  \
0          1  20230101.0         2.0         6.0         86.0       1.0   
1          2  20230101.0         5.0         1.0        156.0       3.0   
2          3  20230101.0         4.0         1.0        145.0       3.0   
3          4  20230101.0         3.0         5.0         86.0       2.0   
4          5  20230101.0         1.0         3.0         86.0       2.0   

   PlantKey  InspectedQuantity  PassedQuantity  DefectiveQuantity  ...  \
0       1.0              359.0           355.0                4.0  ...   
1       1.0              248.0           240.0                8.0  ...   
2       1.0              245.0           238.0                7.0  ...   
3       1.0              216.0           211.0                5.0  ...   
4       1.0              229.0           224.0                5.0  ...   

   InspectionTimeHours InspectorScore ReworkQuantity  ScrapQuantit

**Fact Maintenance**

In [14]:
import pandas as pd
import numpy as np
import random

# ==========================================================
# Manufacturing Analytics
# ViewSonic
# Fact_Maintenance Generator
# ==========================================================

random.seed(42)
np.random.seed(42)

print("="*60)
print("Loading Files...")
print("="*60)

production=pd.read_csv("Fact_Production.csv")
machine=pd.read_csv("Dim_Machine.csv")
employee=pd.read_csv("Dim_Employee.csv")

print("Production :",len(production))
print("Machine :",len(machine))

maintenance=[]

maintenance_id=1

install_year=dict(
zip(
machine.MachineKey,
machine.InstallationYear
)
)

employee_exp=dict(
zip(
employee.EmployeeKey,
employee.ExperienceYears
)
)

maintenance_types=[
"Preventive",
"Corrective",
"Predictive",
"Emergency"
]

status_list=[
"Completed",
"In Progress",
"Scheduled"
]

priority_list=[
"Low",
"Medium",
"High",
"Critical"
]

Loading Files...
Production : 46037
Machine : 38


In [15]:
print("\nGenerating Maintenance Data...")

for _,row in production.iterrows():

    machine_key=row["MachineKey"]

    employee_key=row["EmployeeKey"]

    plant=row["PlantKey"]

    year_installed=install_year[machine_key]

    machine_age=max(1,2026-year_installed)

    downtime=row["DowntimeMinutes"]

    maint_hours=round(
        downtime/60+
        random.uniform(0.5,3),
        2
    )

    labour_hours=round(
        maint_hours*
        random.uniform(0.8,1.2),
        2
    )

    spare_cost=round(

        machine_age*
        random.uniform(35,120),

        2

    )

    labour_cost=round(

        labour_hours*
        random.uniform(45,85),

        2

    )

    total_cost=round(

        spare_cost+

        labour_cost,

        2

    )

    mttr=round(

        maint_hours,

        2

    )

    mtbf=round(

        random.uniform(

            30,

            220

        )/

        machine_age,

        2

    )

    availability=round(

        max(

            70,

            100-

            downtime/10

        ),

        2

    )

    efficiency=round(

        availability*

        random.uniform(

            0.94,

            0.99

        ),

        2

    )

    technician=round(

        min(

            100,

            82+

            employee_exp[employee_key]*0.6+

            random.uniform(-3,4)

        ),

        2

    )

    repeat=random.choice([

        "Yes",

        "No",

        "No",

        "No"

    ])

    maintenance.append({

        "MaintenanceID":maintenance_id,

        "DateKey":row["DateKey"],

        "MachineKey":machine_key,

        "EmployeeKey":employee_key,

        "PlantKey":plant,

        "MaintenanceType":random.choice(maintenance_types),

        "MaintenanceStatus":random.choice(status_list),

        "Priority":random.choice(priority_list),

        "DowntimeMinutes":downtime,

        "MaintenanceDurationHours":maint_hours,

        "LaborHours":labour_hours,

        "SparePartsCost":spare_cost,

        "LaborCost":labour_cost,

        "TotalMaintenanceCost":total_cost,

        "MTTRHours":mttr,

        "MTBFDays":mtbf,

        "MachineAvailabilityPct":availability,

        "MaintenanceEfficiencyPct":efficiency,

        "TechnicianRating":technician,

        "RepeatFailure":repeat

    })

    maintenance_id+=1

maintenance_df=pd.DataFrame(maintenance)

print("Maintenance Records :",len(maintenance_df))


Generating Maintenance Data...
Maintenance Records : 46037


In [16]:
maintenance_df=maintenance_df[[

"MaintenanceID",
"DateKey",
"MachineKey",
"EmployeeKey",
"PlantKey",
"MaintenanceType",
"MaintenanceStatus",
"Priority",
"DowntimeMinutes",
"MaintenanceDurationHours",
"LaborHours",
"SparePartsCost",
"LaborCost",
"TotalMaintenanceCost",
"MTTRHours",
"MTBFDays",
"MachineAvailabilityPct",
"MaintenanceEfficiencyPct",
"TechnicianRating",
"RepeatFailure"

]]

assert maintenance_df.isnull().sum().sum()==0

maintenance_df.to_csv(

"Fact_Maintenance.csv",

index=False

)

print("="*60)
print("Fact_Maintenance.csv Created Successfully")
print("="*60)

print(maintenance_df.head())

Fact_Maintenance.csv Created Successfully
   MaintenanceID     DateKey  MachineKey  EmployeeKey  PlantKey  \
0              1  20230101.0         6.0         86.0       1.0   
1              2  20230101.0         1.0        156.0       1.0   
2              3  20230101.0         1.0        145.0       1.0   
3              4  20230101.0         5.0         86.0       1.0   
4              5  20230101.0         3.0         86.0       1.0   

  MaintenanceType MaintenanceStatus  Priority  DowntimeMinutes  \
0       Emergency         Completed       Low             11.0   
1      Predictive         Completed    Medium             18.0   
2      Predictive         Completed  Critical             39.0   
3      Preventive         Completed    Medium             56.0   
4      Predictive       In Progress    Medium             26.0   

   MaintenanceDurationHours  LaborHours  SparePartsCost  LaborCost  \
0                      2.28        1.85          350.26      99.77   
1                 